# Embeddings & Similarity — From Scratch

**Phase 01 · Notebook 1 of 3**

---

By the end of this notebook you will be able to:

- Explain what an embedding is and why vectors capture meaning
- Embed sentences with `sentence-transformers` and inspect the output shape
- Compute cosine similarity from first principles with NumPy
- Build a semantic search engine with zero external frameworks
- Compare two open-source embedding models side-by-side
- Visualise a high-dimensional embedding space in 2D with UMAP

**Tools used:** `sentence-transformers`, `numpy`, `pandas`, `matplotlib`, `umap-learn`  
**Data:** `../../data/raw/articles.csv` (500 domain articles; auto-generated with `faker` if missing)

---

## Section 1 — What Is an Embedding?

A piece of text — a word, sentence, or full document — cannot be fed directly
into a neural network or compared mathematically. An **embedding** solves this
by mapping text to a fixed-length list of numbers (a vector) that lives in a
high-dimensional space.

The key property is that **meaning is preserved by geometry**: two sentences
that say the same thing (even in different words or languages) will produce
vectors that point in roughly the same direction, while unrelated sentences
will point in very different directions.

This is the fundamental building block of every modern RAG, search, and
recommendation system.

### What to look for
- The CSV columns: `id`, `title`, `abstract`, `category`, `year`
- The six domain categories that will form clusters in Section 6
- A `body` column is synthesised from `abstract` — this is what we embed in Section 4

> **Note:** If `articles.csv` is not present it is generated automatically using
> `faker` so the notebook always runs end-to-end.

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Path helpers ──────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath('')          # directory this notebook lives in
DATA_DIR     = os.path.join(NOTEBOOK_DIR, '..', '..', 'data', 'raw')
DATA_PATH    = os.path.join(DATA_DIR, 'articles.csv')

# ── Auto-generate articles.csv with faker if it does not exist ────────────
if not os.path.exists(DATA_PATH):
    print(f'articles.csv not found at {DATA_PATH}')
    print('Generating 500 synthetic articles with faker...')
    try:
        from faker import Faker
        fake = Faker()
    except ImportError:
        raise SystemExit('Run: pip install faker')

    CATEGORIES = [
        'pension_regulation', 'investment_theory',
        'macroeconomics', 'fintech_ai', 'actuarial', 'general_ml',
    ]

    TITLES = {
        'pension_regulation': [
            'IORP II Implementation Across Member States',
            'FTK Coverage Ratio Requirements Under Dutch Law',
            'Solvency II vs IORP: A Comparative Analysis',
            'Prudent Person Principle in EU Pension Regulation',
            'Own Risk Assessment for Occupational Pensions',
            'Cross-Border Pension Schemes Under IORP III',
            'Governance Requirements for IORPs',
            'Member Communication Standards in Occupational Pensions',
            'Recovery Plans for Underfunded Pension Schemes',
            'DNB Supervisory Framework for Dutch Pension Funds',
        ],
        'investment_theory': [
            'Liability-Driven Investing for Defined Benefit Plans',
            'ESG Integration in Asset-Liability Management',
            'Factor Investing in Pension Fund Portfolios',
            'Alternative Assets and Illiquidity Premium',
            'Dynamic Asset Allocation Under Mean-Variance Framework',
            'Smart Beta Strategies for Long-Horizon Investors',
            'Currency Hedging in International Pension Portfolios',
            'Infrastructure Investment for Pension Funds',
            'Private Equity Allocation in Institutional Portfolios',
            'Tail Risk Hedging with Options Overlays',
        ],
        'macroeconomics': [
            'Inflation and Pension Liability Valuation',
            'Interest Rate Risk in Defined Benefit Plans',
            'Demographic Transition and Pension Sustainability',
            'ECB Policy and Long-Term Bond Yields',
            'Real Asset Returns During Inflationary Periods',
            'Longevity Risk and Population Aging in Europe',
            'Yield Curve Dynamics and Pension Discounting',
            'Macroprudential Policy and Systemic Risk',
            'Housing Markets and Household Pension Savings',
            'Global Supply Chains and Inflation Persistence',
        ],
        'fintech_ai': [
            'Transformer Models for Financial Text Classification',
            'LLMs for Regulatory Document Summarisation',
            'RAG Systems in Financial Compliance',
            'NLP for Earnings Call Analysis',
            'Automated Reporting with Large Language Models',
            'Knowledge Graphs for Financial Regulation',
            'Explainability in ML-Based Credit Scoring',
            'Sentiment Analysis of Central Bank Communications',
            'Algorithmic Trading with Reinforcement Learning',
            'AI Governance Frameworks for Financial Services',
        ],
        'actuarial': [
            'Stochastic ALM for Pension Funds',
            'Mortality Improvement Projections for European Populations',
            'Longevity Swaps as a Risk Transfer Tool',
            'Disability Incidence Trends in Occupational Schemes',
            'Capital Requirements for Defined Benefit Guarantees',
            'Biometric Risk in Multi-Pillar Pension Systems',
            'Cohort vs Period Life Tables: Implications for Valuation',
            'Embedded Value and Market-Consistent Valuation',
            'Run-Off Management of Closed Defined Benefit Schemes',
            'Stress Testing Pension Funds Under Adverse Scenarios',
        ],
        'general_ml': [
            'Machine Learning for Credit Risk Estimation',
            'Gradient Boosting in Financial Forecasting',
            'Time-Series Anomaly Detection in Market Data',
            'Explainable AI for Regulatory Compliance',
            'Federated Learning for Privacy-Preserving Finance',
            'Deep Learning for Fraud Detection',
            'Conformal Prediction in Uncertainty Quantification',
            'Graph Neural Networks for Counterparty Risk',
            'AutoML for Quantitative Analysts',
            'Causal Inference Methods in Economic Research',
        ],
    }

    ABSTRACTS = {
        'pension_regulation': [
            'The IORP II directive introduced new governance and transparency requirements for occupational pension institutions across EU member states, including own-risk assessments and integrated risk management frameworks.',
            'Under the Dutch FTK, pension funds must maintain a policy coverage ratio above 110%. Funds below the minimum required coverage ratio of 90% are required to submit a recovery plan to DNB within three months.',
            'This paper compares the Solvency II framework for insurers with the IORP regime for pension funds, highlighting divergent approaches to capital requirements and risk management obligations.',
            'The prudent person principle requires IORPs to invest solely in the interests of members and beneficiaries, applying a holistic view of risk that goes beyond simple asset-liability matching.',
            'The own-risk assessment (ORA) process mandates that each IORP evaluate its current and future risks, including demographic, market, and operational risks, in a forward-looking manner.',
            'Cross-border pension schemes operating under IORP face complex regulatory fragmentation, as host-state social and labour law requirements often conflict with home-state prudential rules.',
            'Governance under IORP II requires a three-function system: risk management, internal audit, and actuarial, with clear accountability and documentation standards for all key functions.',
            'Pension funds must deliver clear and timely member communications, including a pension benefit statement (PBS) following a standardised template to aid retirement planning decisions.',
            'Underfunded pension schemes in the Netherlands must file recovery plans detailing how they will restore the policy coverage ratio within ten years using realistic return assumptions.',
            'DNB applies a risk-based supervisory model that includes ongoing monitoring, thematic reviews, and on-site inspections, with escalating intervention powers for non-compliant funds.',
        ],
        'investment_theory': [
            'Liability-driven investing (LDI) aligns asset duration with pension liability duration, reducing interest rate and inflation sensitivity and providing more predictable funding outcomes.',
            'ESG criteria can be incorporated into ALM frameworks without materially degrading funding ratio stability, particularly when ESG scores are used as a tilt on factor-based allocations.',
            'Factor investing—targeting value, momentum, quality, and low-volatility premia—offers systematic excess returns for pension funds willing to accept tracking error relative to cap-weighted benchmarks.',
            'Illiquid alternative assets such as infrastructure and private credit offer an illiquidity premium of 100–250bps, providing return enhancement and inflation linkage for long-horizon investors.',
            'Dynamic asset allocation models adjust the equity-bond split in response to funding ratio levels, reducing risk when the fund is underfunded and increasing return seeking when over-funded.',
            'Smart beta strategies harvest well-documented risk premia in a transparent, rule-based manner at lower cost than active management, suitable for pension funds with large allocations.',
            'Currency hedging programs for international equity allocations reduce volatility but introduce roll costs; the optimal hedge ratio depends on liability currency and correlation structure.',
            'Infrastructure equity and debt provide stable, long-duration cash flows with implicit inflation linkage, making them natural matches for defined benefit pension liabilities.',
            'Private equity allocations in pension portfolios can boost long-run returns by 200–400bps but require careful liquidity planning given the J-curve and capital call profile.',
            'Options-based tail risk hedging programs use put spreads or variance swaps to limit left-tail drawdowns, protecting the funding ratio during severe market dislocations.',
        ],
        'macroeconomics': [
            'Rising inflation erodes the real value of nominal pension payments while simultaneously increasing the discount rate, creating a complex net effect on pension liability valuations.',
            'Duration mismatch between pension liabilities and fixed-income assets creates significant interest rate exposure; a 100bps parallel shift can reduce funding ratios by 10–20 percentage points.',
            'Rapid population aging in Western Europe increases old-age dependency ratios, placing pressure on pay-as-you-go state pensions and intensifying the role of funded occupational schemes.',
            'ECB asset purchase programmes have suppressed eurozone long-term yields, compressing the discount rate used to value pension liabilities and worsening funding ratios for defined benefit schemes.',
            'Real assets including commodities, real estate, and inflation-linked bonds have historically maintained real purchasing power during inflationary regimes, providing natural hedges for pension funds.',
            'Longevity improvements of two to three years per decade in Western Europe represent a material risk for defined benefit pension funds that must be explicitly modelled in liability valuations.',
            'The yield curve shape — particularly the slope between 2-year and 30-year government bonds — directly affects the valuation of long-duration pension liabilities and funding ratio dynamics.',
            'Macroprudential tools such as countercyclical capital buffers aim to reduce systemic risk build-up, but pension funds as long-term investors may unintentionally amplify procyclicality.',
            'Household over-reliance on housing wealth as a pension substitute creates concentration risk; falling house prices in retirement can force asset drawdowns at unfavourable valuations.',
            'Supply-chain disruptions and geopolitical fragmentation have introduced structural inflationary pressures that may persist beyond the traditional monetary policy transmission horizon.',
        ],
        'fintech_ai': [
            'We fine-tune BERT on a corpus of financial regulatory documents to classify sections by topic with 94% accuracy, outperforming TF-IDF baselines on pension regulation and investment policy text.',
            'Large language models with retrieval augmentation can summarise hundreds of regulatory pages in seconds, enabling compliance teams to monitor regulatory change across multiple jurisdictions.',
            'RAG systems in financial compliance combine dense retrieval over regulatory corpora with a generative model to answer precise queries about rule applicability, attribution, and exceptions.',
            'NLP analysis of earnings call transcripts using sentiment and topic models reveals forward-looking signals that are incrementally informative over traditional financial statement metrics.',
            'LLM-based automated reporting reduces the time to produce regulatory disclosures from days to hours by extracting structured data from internal systems and drafting narrative commentary.',
            'Knowledge graphs representing regulatory entities, obligations, and cross-references enable complex compliance queries that are difficult to answer with flat document retrieval alone.',
            'Explainability constraints in credit scoring models are increasingly required by EU regulation; SHAP and LIME provide local explanations but may not satisfy the right-to-explanation standard.',
            'Sentiment analysis of central bank communications using large language models captures subtle shifts in forward guidance that precede policy rate changes by several weeks.',
            'Reinforcement learning agents trained on historical order-book data learn execution strategies that reduce market impact, outperforming VWAP and TWAP benchmarks on liquid instruments.',
            'AI governance frameworks for financial services must address model risk, data lineage, fairness, and auditability, with documentation requirements aligned to the EU AI Act risk tiers.',
        ],
        'actuarial': [
            'We develop a stochastic asset-liability model for Dutch pension funds that jointly simulates equity, interest rate, and inflation risks, calibrated to DNB stress scenarios under the FTK.',
            'Mortality improvement projections using the CBD and Lee-Carter models diverge significantly at older ages, creating material uncertainty in the valuation of longevity-contingent liabilities.',
            'Longevity swaps transfer the risk of members living longer than expected to an insurer or capital market counterparty, providing a cost-effective hedge for large pension schemes.',
            'Disability incidence rates in Dutch occupational pension schemes have declined over the past decade, but heterogeneity by sector and age group remains significant for pricing purposes.',
            'Capital requirements for defined benefit guarantees depend on the interaction of interest rate, inflation, and longevity risks; diversification across risk types reduces total capital needs.',
            'Biometric risks — mortality, morbidity, and disability — in multi-pillar pension systems require coordinated modelling across state, occupational, and individual savings pillars.',
            'Cohort life tables that track a birth year population produce systematically lower mortality rates than period tables, with important implications for the valuation of defined benefit liabilities.',
            'Market-consistent embedded value (MCEV) principles apply option pricing techniques to value guarantees embedded in defined benefit and defined contribution pension products.',
            'Run-off management of closed defined benefit schemes involves optimising the investment portfolio to match declining liability cash flows while minimising the cost of residual risk.',
            'Stress tests applying simultaneous shocks to interest rates, equity markets, and longevity reveal the non-linear interaction effects that drive tail risk in pension fund balance sheets.',
        ],
        'general_ml': [
            'Gradient boosting models trained on loan-level data outperform logistic regression for probability of default estimation; macroeconomic feature engineering improves out-of-time stability.',
            'Ensemble methods combining gradient boosting and neural network predictions reduce forecast error in financial time series compared to either model class in isolation.',
            'Isolation forests and autoencoder-based anomaly detectors identify regime changes and data quality issues in high-frequency market data with lower false-positive rates than threshold methods.',
            'Post-hoc explainability methods (SHAP, LIME, Integrated Gradients) differ in their treatment of feature interactions and can yield conflicting attributions for the same model.',
            'Federated learning enables financial institutions to train shared fraud detection models without exchanging customer data, addressing privacy regulation while improving collective detection rates.',
            'Graph neural networks applied to transaction networks identify fraud rings by exploiting structural patterns invisible to feature-based models operating on individual transactions.',
            'Conformal prediction provides distribution-free coverage guarantees for regression and classification outputs, enabling rigorous uncertainty quantification in high-stakes financial decisions.',
            'Graph neural networks applied to bilateral exposure networks learn counterparty risk representations that embed both bilateral and systemic risk signals for default probability estimation.',
            'AutoML pipelines combining hyperparameter optimisation and neural architecture search reduce the time from data to deployment while maintaining model quality for quantitative analysts.',
            'Causal inference methods including difference-in-differences and regression discontinuity are increasingly applied to evaluate the effect of investment policy changes on member outcomes.',
        ],
    }

    random.seed(42)
    rows = []
    article_id = 1
    per_category = 500 // len(CATEGORIES)  # ~83 each

    for cat in CATEGORIES:
        templates_t = TITLES[cat]
        templates_a = ABSTRACTS[cat]
        for i in range(per_category):
            base_title    = templates_t[i % len(templates_t)]
            base_abstract = templates_a[i % len(templates_a)]
            suffix = f' — Study {i // len(templates_t) + 1}' if i >= len(templates_t) else ''
            rows.append({
                'id':             article_id,
                'title':          base_title + suffix,
                'abstract':       base_abstract,
                'category':       cat,
                'year':           random.randint(2018, 2024),
                'authors':        fake.name(),
                'source_journal': fake.company() + ' Journal',
                'word_count':     random.randint(150, 350),
            })
            article_id += 1

    # Top up to exactly 500
    while len(rows) < 500:
        cat = random.choice(CATEGORIES)
        rows.append({
            'id':             article_id,
            'title':          fake.sentence(nb_words=6).rstrip('.'),
            'abstract':       fake.paragraph(nb_sentences=4),
            'category':       cat,
            'year':           random.randint(2018, 2024),
            'authors':        fake.name(),
            'source_journal': fake.company() + ' Journal',
            'word_count':     random.randint(150, 350),
        })
        article_id += 1

    os.makedirs(DATA_DIR, exist_ok=True)
    pd.DataFrame(rows).to_csv(DATA_PATH, index=False)
    print(f'  Saved {len(rows)} articles to {DATA_PATH}')

# ── Load data ─────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} articles  |  columns: {list(df.columns)}')
print(f"Categories : {df['category'].value_counts().to_dict()}")

# ── Synthesise a 'body' column (title + abstract) used in Section 4 ─────
df['body'] = df['title'] + '. ' + df['abstract']

# ── Print first 20 titles ─────────────────────────────────────────────────
sample = df.head(20)
print(f"\nFirst 20 article titles:")
print(f"{'#':<4} {'Category':<22} {'Title'}")
print('-' * 80)
for _, row in sample.iterrows():
    print(f"{row['id']:<4} {row['category']:<22} {row['title'][:52]}")

---

## Section 2 — Your First Embedding

### What is `sentence-transformers`?

`sentence-transformers` is a Python library built on top of Hugging Face
Transformers. It wraps pre-trained BERT-family models that have been
**fine-tuned with contrastive learning** on hundreds of millions of sentence
pairs. The result is a model that produces embeddings where semantic
similarity aligns tightly with geometric proximity.

### The model: `all-MiniLM-L6-v2`

| Property | Value |
|----------|-------|
| Architecture | MiniLM (distilled BERT) — 6 transformer layers |
| Parameters | ~22 million |
| Embedding dimension | **384** |
| Max input length | 256 tokens |
| Download size | ~80 MB (cached after first load) |
| Speed | ~14,000 sentences / second on GPU; ~2,000 on CPU |
| MTEB score (avg) | 56.3 — strong baseline, excellent for its size |

### What does the output shape mean?

Encoding **N** sentences produces a matrix of shape **(N, 384)**:
- **rows** = one embedding per sentence
- **columns** = 384 numbers that together encode the meaning of the sentence

No individual dimension has a human-readable interpretation — meaning emerges
from the *relative position* of vectors in the 384-dimensional space.

### What to look for
- The shape `(20, 384)` confirming 20 vectors of 384 dimensions each
- The L2 norms are close to 1 when `normalize_embeddings=True`
- Two titles that are semantically similar (e.g. two pension regulation titles)
  will have dots ≈ 0.8+, while unrelated pairs will score closer to 0

In [ ]:
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = 'all-MiniLM-L6-v2'
print(f'Loading {MODEL_NAME}...')
model = SentenceTransformer(MODEL_NAME)

print(f'Max sequence length : {model.max_seq_length} tokens')
print(f'Embedding dimension : {model.get_sentence_embedding_dimension()}')

# ── Embed the first 20 titles ─────────────────────────────────────────────
titles_20 = df.head(20)['title'].tolist()

t0 = time.time()
title_embeddings = model.encode(
    titles_20,
    normalize_embeddings=True,  # unit-normalise so dot product == cosine sim
    show_progress_bar=False,
)
elapsed = time.time() - t0

print(f'\nEncoded {len(titles_20)} titles in {elapsed:.2f}s')
print(f'Embedding matrix shape : {title_embeddings.shape}')
print(f'  rows = {title_embeddings.shape[0]}  (one per title)')
print(f'  cols = {title_embeddings.shape[1]}  (embedding dimensions)')
print(f'\nFirst 8 values of title 0:')
print(np.round(title_embeddings[0, :8], 4))
print(f'\nL2 norms (all should be ~1.0 because normalize_embeddings=True):')
norms = np.linalg.norm(title_embeddings, axis=1)
print(f'  min={norms.min():.6f}  max={norms.max():.6f}  mean={norms.mean():.6f}')

---

## Section 3 — Cosine Similarity

### What does cosine similarity measure?

Cosine similarity measures the **angle** between two vectors, not the
distance between their tips. Two vectors pointing in the same direction
score **+1** (identical meaning), perpendicular vectors score **0**
(unrelated), and opposing vectors score **−1** (antonymous).

$$\text{cosine\_sim}(\mathbf{a}, \mathbf{b})
= \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

### Why does dot product work for normalised vectors?

When we call `normalize_embeddings=True`, every vector is scaled to have
magnitude 1 (unit vector). The denominator
$\|\mathbf{a}\|\,\|\mathbf{b}\| = 1 \times 1 = 1$, so:

$$\text{cosine\_sim}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b}$$

This is significant for performance: a batch of dot products is a single
matrix multiplication (`embeddings @ embeddings.T`) — extremely fast.

### Values to expect in practice

| Score range | Interpretation |
|-------------|----------------|
| 0.9 – 1.0 | Near-duplicate sentences |
| 0.7 – 0.9 | Same topic, different phrasing |
| 0.4 – 0.7 | Related domain, different focus |
| 0.0 – 0.4 | Loosely related or unrelated |
| < 0 | Rare; indicates opposing semantics |

### What to look for
- The diagonal is always 1.0 (every vector is identical to itself)
- Articles from the *same category* show brighter off-diagonal squares
- The block structure reveals that the model has separated domains

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Cosine similarity matrix (dot product because vectors are unit-normalised)
sim_matrix = title_embeddings @ title_embeddings.T   # shape (20, 20)

print(f'Similarity matrix shape : {sim_matrix.shape}')
print(f'Diagonal (self-similarity, should all be 1.0):')
print(np.round(np.diag(sim_matrix), 4))

# ── Heatmap ───────────────────────────────────────────────────────────────
short_labels = [t[:30] + '…' if len(t) > 30 else t for t in titles_20]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0.0, vmax=1.0)

ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(short_labels, fontsize=7)

# Annotate each cell with the score
for i in range(20):
    for j in range(20):
        val = sim_matrix[i, j]
        color = 'black' if 0.3 < val < 0.85 else 'white'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=5.5, color=color)

plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title(
    f'Cosine Similarity — First 20 Article Titles\nModel: {MODEL_NAME}',
    fontsize=12, pad=12
)
plt.tight_layout()
plt.savefig('section3_cosine_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nHeatmap saved to section3_cosine_heatmap.png')

---

## Section 4 — Semantic Search From Scratch (No Frameworks)

### Why build this without Chroma or FAISS?

In Section 5 of *Phase 02* you will drop these embeddings into ChromaDB and
enjoy sub-millisecond retrieval over millions of vectors. But if you first
build the same thing with raw NumPy, you will understand:

1. What ChromaDB is actually doing under the hood
2. Why cosine similarity (or dot product) is the right distance metric
3. Why approximate nearest-neighbour (ANN) indices exist — brute-force
   search over 384-D vectors is *O(N × d)* per query, which becomes slow
   beyond ~100k documents
4. What `top_k` means and how scores map to relevance

**This is the foundation of every RAG system.** Master it here.

### What to look for
- Queries about pension regulation should retrieve regulation articles (scores > 0.55)
- Queries about AI/ML should retrieve fintech_ai and general_ml articles
- Off-topic queries will still return results — semantic search has no built-in
  relevance threshold; you need to add one yourself (e.g. `score > 0.4`)
- Embedding 200 bodies should take < 10 seconds on CPU

In [ ]:
# ── Embed all 200 article bodies (title + abstract) ─────────────────────
# We use the first 200 rows to keep CPU time reasonable in a live demo.
# In production you would embed all 500 (or millions) and use an ANN index.

corpus_df = df.head(200).reset_index(drop=True)
bodies     = corpus_df['body'].tolist()

print(f'Embedding {len(bodies)} article bodies with {MODEL_NAME}...')
t0 = time.time()

corpus_embeddings = model.encode(
    bodies,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s  ({len(bodies)/elapsed:.0f} docs/sec)')
print(f'Corpus embedding matrix : {corpus_embeddings.shape}')


# ── Semantic search function ──────────────────────────────────────────────
def search(query: str, top_k: int = 5) -> pd.DataFrame:
    """Return the top_k most similar articles to query."""
    query_vec = model.encode([query], normalize_embeddings=True)[0]
    scores    = corpus_embeddings @ query_vec          # dot product == cosine
    top_idx   = np.argsort(scores)[::-1][:top_k]      # descending sort
    results   = corpus_df.iloc[top_idx][['title', 'category', 'year']].copy()
    results['score'] = np.round(scores[top_idx], 4)
    return results.reset_index(drop=True)


# ── Run 5 example queries ─────────────────────────────────────────────────
queries = [
    'What are the coverage ratio requirements for pension funds?',
    'How do large language models help with regulatory compliance?',
    'How does inflation affect pension liabilities?',
    'What is liability-driven investing?',
    'Explain gradient boosting for financial risk models',
]

for q in queries:
    print(f'\n🔍 Query: "{q}"')
    print(f'   {"#":<3} {"Score":<8} {"Category":<22} {"Title"}')
    print('   ' + '-' * 72)
    results = search(q, top_k=5)
    for rank, row in results.iterrows():
        print(f'   {rank+1:<3} {row["score"]:<8} {row["category"]:<22} {row["title"][:45]}')

---

## Section 5 — Comparing Embedding Models

### Not all models are equal

The [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard)
(Massive Text Embedding Benchmark) provides a standardised comparison of
hundreds of embedding models across 56 tasks. The key insight:

| Model | Dim | Params | Avg MTEB | Relative speed |
|-------|-----|--------|----------|----------------|
| `all-MiniLM-L6-v2` | 384 | 22M | 56.3 | ⚡⚡⚡ fastest |
| `bge-small-en-v1.5` | 384 | 33M | 62.2 | ⚡⚡ fast |
| `bge-base-en-v1.5` | 768 | 109M | 63.6 | ⚡ medium |
| `e5-large-v2` | 1024 | 335M | 64.9 | 🐢 slow |

### The speed vs. quality trade-off

- For **real-time RAG** (< 100ms latency budget), prefer small models like
  `all-MiniLM-L6-v2` or `bge-small-en-v1.5`
- For **offline indexing** where you embed once and search often, a larger
  model pays back its cost many times over in retrieval quality
- **Domain fine-tuning** (Phase 04) typically adds more value than switching
  to a larger general-purpose model

### What to look for
- `bge-small-en-v1.5` often returns slightly higher scores and tighter results
  for the regulation queries — it was fine-tuned specifically for retrieval
- The rank order of results frequently differs between the two models
- Neither model is always better; the best choice depends on your domain
  and the MTEB sub-tasks that most resemble your use case

In [ ]:
MODEL_B_NAME = 'BAAI/bge-small-en-v1.5'
print(f'Loading {MODEL_B_NAME}...')
model_b = SentenceTransformer(MODEL_B_NAME)

# ── Embed corpus with model B ──────────────────────────────────────────────
print(f'Embedding {len(bodies)} bodies with {MODEL_B_NAME}...')
t0 = time.time()
corpus_embeddings_b = model_b.encode(
    bodies,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f'Done in {time.time()-t0:.1f}s  |  shape: {corpus_embeddings_b.shape}')


def search_b(query: str, top_k: int = 5) -> pd.DataFrame:
    """Search using bge-small-en-v1.5."""
    # BGE models perform best with an instruction prefix for retrieval queries
    prefixed = f'Represent this sentence for retrieval: {query}'
    query_vec = model_b.encode([prefixed], normalize_embeddings=True)[0]
    scores    = corpus_embeddings_b @ query_vec
    top_idx   = np.argsort(scores)[::-1][:top_k]
    results   = corpus_df.iloc[top_idx][['title', 'category', 'year']].copy()
    results['score'] = np.round(scores[top_idx], 4)
    return results.reset_index(drop=True)


# ── Side-by-side comparison ────────────────────────────────────────────────
WIDTH = 45
for q in queries:
    r_a = search(q, top_k=3)
    r_b = search_b(q, top_k=3)
    print(f'\n🔍 Query: "{q}"')
    print(f'  {"all-MiniLM-L6-v2":<{WIDTH+8}}  {"bge-small-en-v1.5"}')
    print('  ' + '-' * (WIDTH * 2 + 12))
    for i in range(3):
        a_str = f'[{r_a.iloc[i]["score"]}] {r_a.iloc[i]["title"][:WIDTH]}'
        b_str = f'[{r_b.iloc[i]["score"]}] {r_b.iloc[i]["title"][:WIDTH]}'
        print(f'  {a_str:<{WIDTH+8}}  {b_str}')

---

## Section 6 — Visualising the Embedding Space

### Why visualise embeddings?

Embeddings live in 384 dimensions. That is impossible to perceive directly.
**UMAP** (Uniform Manifold Approximation and Projection) is a dimensionality
reduction algorithm that projects the 384-D cloud down to 2-D while
preserving **local structure** — points that were close together in high
dimensions stay close together in the plot.

Compared to PCA (which maximises global variance) and t-SNE (which is slow
and stochastic), UMAP is:
- **Faster** — scales to millions of points
- **Better at preserving local topology** — clusters are more compact
- **Reproducible** — with a fixed `random_state`

### What to look for
- Tight clusters by category confirm the model has separated domains
- Overlapping clusters (e.g. `actuarial` and `pension_regulation`) mean the
  model sees semantic overlap between those domains — which is correct!
- If categories are totally mixed, the embedding model or data quality
  may need improvement
- Run this cell a second time with a different `n_neighbors` value (try 5
  vs 50) to see how local vs global structure changes the picture

> **Tip:** The `random_state=42` makes the layout reproducible. Remove it
> and each run will produce a different (but valid) layout.

In [ ]:
# Install umap-learn if not already present
import importlib
if importlib.util.find_spec('umap') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'umap-learn', '--quiet'])

import umap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Embed all 500 articles (or however many are in df) ────────────────────
all_bodies = df['body'].tolist()
print(f'Embedding all {len(all_bodies)} articles for UMAP visualisation...')
t0 = time.time()
all_embeddings = model.encode(
    all_bodies,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f'Embedded in {time.time()-t0:.1f}s  |  shape: {all_embeddings.shape}')

# ── UMAP reduction 384-D → 2-D ────────────────────────────────────────────
print('Running UMAP (this may take ~30s on CPU)...')
t0 = time.time()
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,      # local neighbourhood size; smaller = finer clusters
    min_dist=0.1,        # minimum distance in 2-D layout
    metric='cosine',     # use cosine distance to match our similarity metric
    random_state=42,
)
coords_2d = reducer.fit_transform(all_embeddings)
print(f'UMAP done in {time.time()-t0:.1f}s  |  output shape: {coords_2d.shape}')

# ── Scatter plot ──────────────────────────────────────────────────────────
categories = df['category'].unique()
palette    = sns.color_palette('tab10', n_colors=len(categories))
colour_map = dict(zip(categories, palette))

fig, ax = plt.subplots(figsize=(11, 8))

for cat in categories:
    mask = df['category'] == cat
    ax.scatter(
        coords_2d[mask, 0], coords_2d[mask, 1],
        c=[colour_map[cat]], label=cat,
        s=50, alpha=0.75, edgecolors='white', linewidths=0.3,
    )

ax.set_xlabel('UMAP dimension 1', fontsize=11)
ax.set_ylabel('UMAP dimension 2', fontsize=11)
ax.set_title(
    f'Embedding Space — {len(df)} Articles by Category (UMAP 2D)\n'
    f'Model: {MODEL_NAME}  |  n_neighbors=15  |  metric=cosine',
    fontsize=12
)
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('section6_umap.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nUMAP plot saved to section6_umap.png')

---

## ✅ Summary — What You Learned

| # | Concept | Key takeaway |
|---|---------|-------------|
| 1 | Embeddings | Text → fixed-length vector; meaning encoded as geometry |
| 2 | `sentence-transformers` | `all-MiniLM-L6-v2` gives 384-D unit vectors in < 1ms/sentence |
| 3 | Cosine similarity | Dot product on unit vectors; range [−1, 1]; diagonal = 1 |
| 4 | Semantic search | `embed → score → argsort` — the core of every RAG retriever |
| 5 | Model comparison | `bge-small-en-v1.5` scores higher on MTEB; always benchmark |
| 6 | UMAP | 384-D → 2-D; tight clusters confirm domain separation |

### Key numbers to remember

- `all-MiniLM-L6-v2` → **384 dimensions**, ~22M params, ~80 MB
- Scores > **0.7** are reliably semantically similar for this model
- Brute-force search over 200 docs: **< 1ms** per query
- At 10M docs the same brute-force would take ~10 seconds — this is why
  vector databases like ChromaDB use approximate nearest-neighbour (ANN) indices

### What comes next — Phase 02: LLMs, Prompting & First RAG

In the next phase you will:

1. **Run a local LLM** via Ollama (`llama3.1:8b`) — no cloud, no keys
2. **Master prompt engineering** patterns (zero-shot, few-shot, CoT, structured output)
3. **Ingest documents** into ChromaDB — the production-grade version of what
   you built from scratch in Section 4
4. **Wire the full RAG chain** with LangChain: query → retrieve → prompt → generate
5. **Inspect** retrieved context and generated answers side-by-side

The embedding intuition you built here is the foundation for everything that follows.